# Task 3 - Model Explainability with SHAP
Understanding WHY the model flags transactions as fraud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os
import sys
sys.path.insert(0, '..')
from sklearn.model_selection import train_test_split
from src.data_processing import (
    load_data, clean_fraud_data, clean_credit_data,
    merge_geolocation, engineer_features,
    transform_features, handle_imbalance
)
os.makedirs('../data/processed', exist_ok=True)
print('Libraries loaded!')

## Load Trained Models and Data

In [ ]:
# Load trained models
rf_fraud = joblib.load('../models/rf_fraud.pkl')
rf_credit = joblib.load('../models/rf_credit.pkl')
print('Models loaded!')
print(f'RF Fraud: {rf_fraud}')
print(f'RF Credit: {rf_credit}')

In [ ]:
# Prepare fraud data
fraud_df, ip_df, credit_df = load_data()
fraud_df = clean_fraud_data(fraud_df)
fraud_df = merge_geolocation(fraud_df, ip_df)
fraud_df = engineer_features(fraud_df)
fraud_df = transform_features(fraud_df, 'class')
X_f = fraud_df.drop(columns=['class'])
y_f = fraud_df['class']
X_f_train, X_f_test, y_f_train, y_f_test = train_test_split(
    X_f, y_f, test_size=0.2, random_state=42, stratify=y_f
)
print(f'Fraud test set: {X_f_test.shape}')

In [ ]:
# Prepare credit data
credit_df = clean_credit_data(credit_df)
X_c = credit_df.drop(columns=['Class'])
y_c = credit_df['Class']
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)
print(f'Credit test set: {X_c_test.shape}')

## Part 1 - Built-in Feature Importance

In [ ]:
# Built-in feature importance for Fraud dataset
importance_f = pd.DataFrame({
    'feature': X_f_test.columns,
    'importance': rf_fraud.feature_importances_
}).sort_values('importance', ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_f, x='importance', y='feature', color='steelblue')
plt.title('Top 10 Features - Built-in Importance (E-Commerce Fraud)')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance_fraud.png')
plt.show()
print(importance_f)

In [ ]:
# Built-in feature importance for Credit dataset
importance_c = pd.DataFrame({
    'feature': X_c_test.columns,
    'importance': rf_credit.feature_importances_
}).sort_values('importance', ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_c, x='importance', y='feature', color='steelblue')
plt.title('Top 10 Features - Built-in Importance (Credit Card)')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance_credit.png')
plt.show()
print(importance_c)

## Part 2 - SHAP Explainability for E-Commerce Fraud

In [ ]:
# Use small sample for speed
X_f_sample = X_f_test.sample(500, random_state=42)
print('Computing SHAP values for Fraud dataset...')
explainer_f = shap.TreeExplainer(rf_fraud)
shap_values_f = explainer_f.shap_values(X_f_sample)
print('SHAP values computed!')

In [ ]:
# SHAP Summary Plot - Fraud
plt.figure()
shap.summary_plot(
    shap_values_f[1],
    X_f_sample,
    plot_type='bar',
    max_display=10,
    show=False
)
plt.title('SHAP Summary Plot - E-Commerce Fraud')
plt.tight_layout()
plt.savefig('../data/processed/shap_summary_fraud.png')
plt.show()
print('SHAP summary plot saved!')

In [ ]:
# SHAP Force Plots - 3 individual predictions
y_f_sample = y_f_test.loc[X_f_sample.index]
y_pred_f = rf_fraud.predict(X_f_sample)

# Find True Positive, False Positive, False Negative
tp_idx = X_f_sample[(y_f_sample.values == 1) & (y_pred_f == 1)].index[0]
fp_idx = X_f_sample[(y_f_sample.values == 0) & (y_pred_f == 1)].index[0]
fn_idx = X_f_sample[(y_f_sample.values == 1) & (y_pred_f == 0)].index[0]
print(f'True Positive index: {tp_idx}')
print(f'False Positive index: {fp_idx}')
print(f'False Negative index: {fn_idx}')

In [ ]:
# Force plot - True Positive
tp_pos = X_f_sample.index.get_loc(tp_idx)
shap.force_plot(
    explainer_f.expected_value[1],
    shap_values_f[1][tp_pos],
    X_f_sample.iloc[tp_pos],
    matplotlib=True,
    show=False
)
plt.title('Force Plot - True Positive (Fraud correctly caught)')
plt.tight_layout()
plt.savefig('../data/processed/shap_force_tp.png', bbox_inches='tight')
plt.show()
print('True Positive force plot saved!')

In [ ]:
# Force plot - False Positive
fp_pos = X_f_sample.index.get_loc(fp_idx)
shap.force_plot(
    explainer_f.expected_value[1],
    shap_values_f[1][fp_pos],
    X_f_sample.iloc[fp_pos],
    matplotlib=True,
    show=False
)
plt.title('Force Plot - False Positive (Legitimate flagged as Fraud)')
plt.tight_layout()
plt.savefig('../data/processed/shap_force_fp.png', bbox_inches='tight')
plt.show()
print('False Positive force plot saved!')

In [ ]:
# Force plot - False Negative
fn_pos = X_f_sample.index.get_loc(fn_idx)
shap.force_plot(
    explainer_f.expected_value[1],
    shap_values_f[1][fn_pos],
    X_f_sample.iloc[fn_pos],
    matplotlib=True,
    show=False
)
plt.title('Force Plot - False Negative (Fraud missed by model)')
plt.tight_layout()
plt.savefig('../data/processed/shap_force_fn.png', bbox_inches='tight')
plt.show()
print('False Negative force plot saved!')

## Part 3 - SHAP for Credit Card Dataset

In [ ]:
X_c_sample = X_c_test.sample(500, random_state=42)
print('Computing SHAP values for Credit Card dataset...')
explainer_c = shap.TreeExplainer(rf_credit)
shap_values_c = explainer_c.shap_values(X_c_sample)
print('SHAP values computed!')

In [ ]:
# SHAP Summary Plot - Credit
plt.figure()
shap.summary_plot(
    shap_values_c[1],
    X_c_sample,
    plot_type='bar',
    max_display=10,
    show=False
)
plt.title('SHAP Summary Plot - Credit Card Fraud')
plt.tight_layout()
plt.savefig('../data/processed/shap_summary_credit.png')
plt.show()
print('Credit SHAP summary saved!')

## Part 4 - SHAP vs Built-in Feature Importance Comparison

In [ ]:
# Compare SHAP vs built-in for Fraud dataset
shap_importance_f = pd.DataFrame({
    'feature': X_f_sample.columns,
    'shap_importance': np.abs(shap_values_f[1]).mean(axis=0)
}).sort_values('shap_importance', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(
    data=importance_f, x='importance', y='feature',
    ax=axes[0], color='steelblue'
)
axes[0].set_title('Built-in Feature Importance')
sns.barplot(
    data=shap_importance_f, x='shap_importance', y='feature',
    ax=axes[1], color='coral'
)
axes[1].set_title('SHAP Feature Importance')
plt.suptitle('Built-in vs SHAP Importance - E-Commerce Fraud')
plt.tight_layout()
plt.savefig('../data/processed/shap_vs_builtin_fraud.png')
plt.show()
print('Comparison plot saved!')

## Part 5 - Top 5 Fraud Drivers and Business Recommendations

In [ ]:
print('=== TOP 5 FRAUD PREDICTION DRIVERS ===')
print(shap_importance_f.head(5).to_string(index=False))
print()
print('=== BUSINESS RECOMMENDATIONS ===')
print()
print('1. TIME SINCE SIGNUP')
print('   SHAP shows new accounts (low time_since_signup) are highest fraud risk.')
print('   RECOMMENDATION: Add extra verification for purchases within 24 hours of signup.')
print()
print('2. PURCHASE VALUE')
print('   High purchase values strongly predict fraud.')
print('   RECOMMENDATION: Flag transactions above 2x the customer average for review.')
print()
print('3. TRANSACTION VELOCITY')
print('   Multiple rapid transactions predict fraud.')
print('   RECOMMENDATION: Block accounts making 3+ purchases within 10 minutes.')
print()
print('4. HOUR OF DAY')
print('   SHAP shows late night transactions (1am-4am) carry higher fraud risk.')
print('   RECOMMENDATION: Apply stricter thresholds for transactions between 1am-4am.')
print()
print('5. COUNTRY ORIGIN')
print('   Transactions from high-risk countries have strong SHAP fraud signal.')
print('   RECOMMENDATION: Use country-based risk tiers to adjust approval thresholds.')